# Interview Prep — Senior Software Engineer, Platform & Agentic AI Engineering (The Hartford)
### HackerRank-powered hiring test | 8+ yrs IT, 2.5 yrs Agentic AI

---

## 0. Role snapshot — how it maps to what you already know

The JD (Hartford, CT / Columbus, OH / Chicago, IL / Charlotte, NC — hybrid Tue–Thu) asks for:

| JD requirement | Your current position |
|---|---|
| LangGraph / LangChain orchestration (LCEL, stateful graphs, checkpointing, HITL, memory, retrievers, callbacks, LangSmith) | **Strong** — you already work with LangGraph/LangChain and multi-agent patterns (ReAct, Planner-Executor, Manager-Worker). LangSmith tracing/eval and LCEL specifics are worth a quick refresh. |
| MCP (Model Context Protocol) | **Strong** — you've done MCP server development already. |
| Google ADK (Agent Development Kit) | **Gap.** You know Bedrock/GCP broadly but likely haven't touched ADK specifically — study this first. |
| RAG | **Strong** — ChromaDB/FAISS/Pinecone experience covers this well. |
| GraphRAG (knowledge-graph-augmented retrieval) | **Gap.** Different from vector RAG — needs targeted prep. |
| AlloyDB AI/Agentic capabilities (vector indexing, Vertex AI integration, agent memory stores) | **Gap.** You know vector DBs conceptually; AlloyDB specifically is new — prep the concept, not necessarily hands-on depth. |
| GCP (Vertex AI, Cloud Run, Cloud Storage) | **Partial** — you have GCP exposure; Vertex AI/Cloud Run specifics worth reviewing. |
| Terraform / IaC | **Gap** if you haven't used it — flag this honestly if asked, but understand the concept. |
| Agent harness internals (execution loop, tool-call orchestration, token-budget mgmt, guardrails) | **Strong conceptually** from your agentic AI work — be ready to describe this in engineering detail, not just at a high level. |
| Python/TypeScript, secure distributed systems | **Strong** on Python/.NET side; TypeScript less certain — mention Python as primary. |

**Interview strategy:** Lead with your strongest area (MCP + LangGraph + multi-agent orchestration + RAG), be honest but structured about ADK/Terraform/AlloyDB/GraphRAG gaps ("I haven't used X directly, but here's the analogous thing I have used, and here's how I'd ramp up"), and make sure you can go deep — not just definitions — on agent harness internals, since that's clearly a core focus of this specific role.

---

## 1. MCP & Agentic Protocols (core focus area)

**Q1. What problem does MCP solve that a normal REST API or LangChain tool wrapper doesn't?**
A: MCP standardizes *how* an LLM discovers and invokes external tools/resources across any client, instead of every app writing bespoke tool-calling glue. It defines a uniform protocol for capability registration (tools, resources, prompts), authentication, and structured request/response — so an agent built against one MCP server works the same way against any other MCP server, and a single MCP server can serve many different agent clients (Claude, custom agents, IDEs) without rewriting integration code per client.

**Q2. Walk through the MCP request lifecycle for a tool call.**
A: Client connects to server → server advertises capabilities (`tools/list`, `resources/list`) → client (or the LLM through the client) selects a tool and calls `tools/call` with structured JSON arguments matching the tool's schema → server executes, returns a structured result (often JSON or text content blocks) → client feeds the result back into the model's context as a tool result message, and the loop continues.

**Q3. How do you secure tool invocation in an MCP server?**
A: Scope credentials per-connection (not global secrets baked into the tool code), validate/sanitize all incoming arguments against the tool's schema before execution, apply least-privilege on any downstream API/DB the tool touches, log/audit every tool call for traceability, and add explicit allow-lists for destructive operations (writes/deletes) rather than trusting the model's judgment alone — treat the LLM's tool call intent as untrusted input.

**Q4. Session state and memory in MCP — how would you design it?**
A: Keep session state server-side, keyed by a session/connection ID, separate from the LLM's own context window — e.g., a Redis or Postgres-backed store holding conversation state, tool-call history, and any long-lived agent memory. The MCP server exposes read/write primitives for that state as resources/tools rather than relying on the model to "remember" across turns.

**Q5. What is Google ADK, and how does it differ from LangGraph or MCP conceptually?**
A: ADK (Agent Development Kit) is Google's framework for building, evaluating, and deploying agents — it gives you agent definition, tool integration, orchestration (including multi-agent hierarchies), and built-in evaluation, with tight integration into Vertex AI for deployment. Compared to LangGraph (which is primarily an orchestration/state-graph library you assemble yourself) ADK is more opinionated/batteries-included, especially for GCP-native deployment. MCP, in contrast, isn't an orchestration framework at all — it's a protocol for tool/resource access that ADK or LangGraph-based agents can both consume. *(If you haven't used ADK hands-on, be upfront: "I haven't shipped with ADK specifically, but I've done equivalent orchestration with LangGraph, and my understanding is ADK's differentiator is native Vertex AI deployment and built-in eval — I'd want to get hands-on with it quickly.")*

**Q6. Compare MCP vs. ADK's Agent2Agent (A2A)-style protocols.**
A: MCP is agent-to-tool/resource (vertical: how one agent reaches external capabilities). A2A-style protocols are agent-to-agent (horizontal: how independent agents discover each other's capabilities and delegate/negotiate tasks). A production agentic platform typically needs both: MCP for tool access, A2A-style patterns for multi-agent delegation across team/org boundaries.

---

## 2. LangGraph / LangChain deep dive

**Q7. Why use LangGraph instead of a plain LangChain chain for an agent?**
A: Chains are DAGs of one-directional steps — fine for linear pipelines. Agents need cycles (reason → act → observe → reason again), conditional branching, and persistent state across steps. LangGraph models this explicitly as a graph with nodes (steps) and edges (including conditional edges), gives you a shared, typed state object threaded through the whole run, and adds checkpointing so you can pause/resume/replay execution — which a plain chain can't do.

**Q8. What is checkpointing in LangGraph and why does it matter for production agents?**
A: Checkpointing persists the graph's state after each step (to memory, SQLite, Postgres, etc.), so you can: resume a long-running or interrupted agent run from the last good state, implement human-in-the-loop (pause before a sensitive tool call, wait for approval, resume), and get full replay/debuggability of exactly how an agent reached a decision — critical for an insurance company's audit/compliance needs.

**Q9. How would you implement human-in-the-loop approval before an agent takes a high-risk action (e.g., issuing a payout)?**
A: Add an explicit interrupt point in the graph before the sensitive tool node — LangGraph supports `interrupt_before`/breakpoints tied to checkpointing. The graph pauses, state is persisted, a human reviews the proposed action (visible in the checkpoint state), and on approval the graph resumes from that exact point rather than restarting the whole run.

**Q10. What's LCEL, and where would you use it vs. a full LangGraph state machine?**
A: LCEL (LangChain Expression Language) is the `|`-pipe composition syntax for chaining runnables (prompt → model → parser, etc.) — good for straightforward, mostly-linear flows with built-in streaming/batching/async support. You'd reach for LangGraph instead when you need branching, loops, multi-agent handoff, or persisted state — i.e., anything that isn't a straight pipeline.

**Q11. How do you manage token-budget/context window across a long multi-turn agent session?**
A: Summarize/compact older conversation turns instead of keeping full history verbatim, use a retriever to pull only relevant memory back in on demand (RAG-as-memory) rather than stuffing everything into context, cap tool-result sizes before they re-enter context, and track running token counts so you can proactively trim before hitting the model's limit rather than failing on overflow.

**Q12. What's the role of callbacks/LangSmith tracing in a production agent system?**
A: Callbacks hook into every step of a run (LLM call start/end, tool call, chain step) for logging, cost tracking, and streaming intermediate output to a UI. LangSmith specifically gives you full trace visualization of a run (every prompt, tool call, and intermediate state), plus evaluation datasets to regression-test agent behavior when you change a prompt or swap a model — this is your main tool for debugging *why* an agent did something wrong in production.

---

## 3. RAG & GraphRAG

**Q13. Standard RAG pipeline — walk through it end to end.**
A: Ingest documents → chunk (with overlap, respecting semantic boundaries where possible) → embed each chunk → store in a vector DB with metadata → at query time, embed the user query → similarity search (often + metadata filtering) → optionally rerank top-k results → inject the retrieved chunks into the prompt as context → generate the answer, ideally with citations back to source chunks.

**Q14. What are the common failure modes of RAG, and how do you mitigate each?**
A: (1) Chunking splits semantic meaning across boundaries → use semantic/recursive chunking with overlap. (2) Query-chunk vocabulary mismatch → hybrid search (dense + BM25/keyword) or query rewriting/expansion. (3) Irrelevant top-k results diluting the answer → add a reranker (cross-encoder) after initial retrieval. (4) Stale index vs. source data → incremental re-indexing pipelines with change detection. (5) Hallucination despite retrieved context → prompt the model to answer only from provided context and cite sources, and evaluate faithfulness explicitly.

**Q15. What is GraphRAG and when would you reach for it over vector RAG?**
A: GraphRAG builds a knowledge graph from the corpus (entities + relationships extracted via LLM), then answers queries by traversing/reasoning over that graph structure rather than (or in addition to) pure vector similarity. It's stronger for multi-hop questions that require connecting facts across documents ("which claims adjusters handled policies underwritten by X in region Y") — something flat vector similarity struggles with because the answer isn't semantically similar to any single chunk, it's derived from relationships between chunks/entities.

**Q16. How would AlloyDB fit into a RAG architecture, and why would The Hartford pick it over a dedicated vector DB like Pinecone?**
A: AlloyDB is Postgres-compatible with a vector extension (pgvector-based) and tight Vertex AI integration for embeddings, so you get vector search *and* your structured transactional/relational data (policies, claims, customers) in the same database — meaning you can do hybrid retrieval that joins vector similarity with relational filters/joins in a single query, and you avoid running/syncing a separate vector store. That's attractive for an enterprise like an insurer where retrieval context often needs to be joined against structured business data (e.g., "find similar claims descriptions AND filter to this policyholder's active policies").

---

## 4. Agent harness internals (be ready to go deep here — this role explicitly builds harnesses, not just uses frameworks)

**Q17. Describe the core agent execution loop from scratch (framework-agnostic).**
A: (1) Assemble context — system prompt + relevant memory/RAG results + available tool schemas + conversation history (trimmed to budget). (2) Call the LLM. (3) Parse the response — is it a final answer, or a tool call request? (4) If tool call: validate arguments against schema, execute the tool (with timeout/error handling), append the tool result to context. (5) Loop back to (2) until a stopping condition (final answer, max iterations, or an error/guardrail trip). (6) Return the final result and log the full trace.

**Q18. How do you prevent infinite loops or runaway tool-calling in an agent?**
A: Hard cap on iterations/tool calls per run, cost/token budget ceiling that terminates the run when exceeded, loop-detection (same tool + same args repeated N times → abort or escalate to human), and timeouts on both individual tool calls and the overall run.

**Q19. How do you design sub-agent delegation (Manager-Worker / Planner-Executor)?**
A: A manager/planner agent decomposes the task into subtasks, and hands each to a specialized worker agent (with its own scoped tools/context) — either sequentially or in parallel where subtasks are independent. Results are collected back into the manager's context for synthesis. Key design points: keep each worker's context minimal and scoped to its subtask (don't just pass the entire parent context down — that blows the budget), define a clear contract/schema for what a worker returns, and decide explicitly whether workers can call tools directly or must request the manager do so (affects guardrail enforcement point).

**Q20. What guardrails would you put around an agent that has access to sensitive systems (e.g., an insurance claims system)?**
A: Input guardrails (validate/sanitize what reaches the model — PII redaction, prompt-injection detection on any external content fed into context), output guardrails (schema-validate tool-call arguments before execution; block or require approval for destructive/high-value actions above a threshold), and a hard separation between "read" tools (agent can call freely) and "write" tools (require human-in-the-loop approval or run in a sandboxed/staging mode first). Also: full audit logging of every tool call with the reasoning trace, since this is a regulated industry.

**Q21. How do you handle streaming output from an agent that's also making tool calls mid-stream?**
A: Stream token-by-token for the natural-language portions of the response, but buffer and don't stream partial tool-call JSON (you need the complete, valid JSON before executing) — most frameworks expose this as distinct event types (`on_llm_new_token` vs. `on_tool_start`/`on_tool_end`) so the UI can render "thinking/talking" tokens live while showing a distinct "calling tool X..." indicator during tool execution.

---

## 5. GCP / Platform Engineering

**Q22. Vertex AI vs. Bedrock — since you know Bedrock, be ready to translate.**
A: Both are managed platforms for hosting/calling foundation models plus fine-tuning, evaluation, and deployment tooling. Vertex AI is GCP's equivalent of Bedrock — Vertex AI Agent Builder / ADK for agent-specific tooling, Vertex AI endpoints for model serving, Vertex AI Search for managed RAG. Cloud Run is GCP's serverless container platform (comparable to AWS Fargate/Lambda-for-containers) — a natural place to deploy an agent microservice or MCP server since it scales to zero and handles HTTP/streaming well.

**Q23. What is Terraform and why does an AI platform team need it?**
A: Terraform is infrastructure-as-code — you declare your infra (Cloud Run services, AlloyDB instances, IAM roles, networking) in config files, and Terraform plans/applies changes idempotently, giving you version-controlled, reviewable, repeatable infra instead of manual console clicks. For an AI platform team, this matters because agent infrastructure (vector DBs, model endpoints, secrets, network policies for tool access) needs to be reproducible across dev/staging/prod and auditable for a regulated company. *(If you're new to Terraform: "I haven't used Terraform hands-on, but I understand IaC principles from [whatever you've used — ARM templates/CloudFormation/manual GCP if any] and would ramp up quickly given the parallels.")*

**Q24. How would you design a secure, scalable AI microservice on Cloud Run that wraps an agent?**
A: Stateless container (state lives in AlloyDB/Redis, not in-memory) so Cloud Run can scale horizontally and cold-start cleanly; secrets via Secret Manager, not env vars in the image; IAM-scoped service account with least-privilege access to only the GCP resources it needs (Vertex AI, AlloyDB, Cloud Storage); request-level auth (validate caller identity before invoking any tool with side effects); and structured logging/tracing (Cloud Logging + OpenTelemetry) so every agent run is debuggable in production.

---

## 6. LLM Safety, Governance, Context Window, Prompt Engineering

**Q25. How do you defend an agent against prompt injection from retrieved/external content?**
A: Treat all external content (retrieved documents, tool results, web pages) as untrusted data, never as instructions — structurally separate "system instructions" from "external content" in the prompt (e.g., clear delimiters, or a model that distinguishes instruction vs. data channels). Add a guardrail/classifier step that flags content attempting to redirect the agent's behavior, and never let a tool result alone trigger another sensitive tool call without validation.

**Q26. What does "context window management" mean practically, and what breaks if you get it wrong?**
A: Practically: deciding what goes into the limited context on every single LLM call — system prompt, relevant memory, retrieved docs, conversation history, tool schemas — and trimming/summarizing to fit. Get it wrong and you either hit hard errors (context overflow), silently truncate important information (agent "forgets" earlier constraints), or pay for and slow down every call with irrelevant bloat that also dilutes the model's attention on what actually matters.

**Q27. How would you evaluate/govern an agent before it goes to production at an insurance company?**
A: Build an eval dataset of representative + adversarial tasks (including prompt-injection attempts and edge cases specific to insurance workflows), track task success rate, tool-call correctness, and safety-guardrail trip rate over that set, gate deployment on regression thresholds (LangSmith or a custom harness), and keep human-in-the-loop on any action with real financial/legal consequence until the agent has a proven track record — plus ongoing production monitoring, not just pre-launch eval.

**Q28. Give an example of a prompt-engineering technique you'd use to reduce hallucination in a RAG answer.**
A: Explicitly instruct the model to answer only using the provided context and to say "I don't have enough information" rather than guessing; ask for inline citations tied to specific retrieved chunks (forces grounding); and use a structured output format (e.g., JSON with an `answer` + `sources` + `confidence` field) so downstream code can flag low-confidence or uncited answers for review instead of surfacing them directly.

---

## 7. Likely coding round (HackerRank Python)

Given the role, expect a mix of general DSA fundamentals *and* practical agentic/platform-flavored coding (parsing structured LLM outputs, building a small tool-calling loop, rate limiting, retry logic). Practice these patterns specifically:

- **Parsing/validating structured LLM output** — write a function that takes a raw string, extracts JSON (handling the model wrapping it in markdown fences or adding preamble text), validates it against a schema, and raises a clear error on malformance.
- **A minimal tool-calling loop** — given a mock "LLM" function that returns either `{"final_answer": ...}` or `{"tool_call": {"name": ..., "args": {...}}}`, write the loop that dispatches to a tool registry (dict of name → function), feeds the result back, and terminates on max iterations or final answer.
- **Retry with exponential backoff + jitter** — for calling a flaky external API (a realistic "tool call may fail" scenario).
- **Simple rate limiter / token-bucket** — since token-budget management is explicitly in the JD.
- **Standard DSA**: sliding window, two pointers, BFS/DFS on graphs (relevant since GraphRAG involves graph traversal!), and dictionary/hash-map-heavy problems (parsing/aggregation) — these show up disproportionately often in practical/agentic-flavored HackerRank sets.

---

## 8. System design prompt to rehearse

**"Design a multi-agent system that lets a Hartford claims adjuster ask natural-language questions across policy documents, claim history, and internal guidelines, with a human approval step before any claim status is updated."**

Structure your answer around:
1. **Ingestion & retrieval layer** — RAG over policy docs/guidelines (vector + AlloyDB hybrid), structured claims data queried directly (not RAG'd).
2. **Orchestration** — LangGraph state machine: planner agent decomposes the question → retriever/query agents run in parallel → synthesis agent composes the answer.
3. **Tool layer via MCP** — claims-system read tools, document-search tools, and a *write* tool (`update_claim_status`) gated behind an explicit `interrupt_before` human-approval checkpoint.
4. **Guardrails** — PII handling, audit logging of every tool call and the reasoning trace, hard iteration/cost caps.
5. **Deployment** — Cloud Run microservice per agent role (or one service hosting the graph), AlloyDB for memory/state + vector search, Terraform-managed infra, Vertex AI for model hosting.
6. **Eval/monitoring** — LangSmith traces in dev, production monitoring on guardrail trip rate and task success.

Practice saying this out loud in under 3–4 minutes — that's usually the pacing for a system-design portion of a HackerRank-style loop.

---

## 9. Behavioral questions (tailored to your 8+ yrs / 2.5 yrs agentic profile)

- *"Walk me through a real coordination or failure incident in one of your agentic AI projects at Cognizant, and how you fixed it."* — Have one concrete story ready (this is the exact kind of story Claude previously suggested you document for your research angle too — reuse it here).
- *"You have 8 years in .NET/Angular and 2.5 in agentic AI — how do you decide when to reach for an agent vs. a simpler deterministic pipeline?"* — Good answer: agents earn their complexity when the task requires dynamic tool selection/branching based on intermediate results; if the workflow is a fixed sequence, a deterministic pipeline is cheaper, faster, and easier to audit — especially relevant to say out loud given this is an insurance company that will care about auditability.
- *"Tell me about a time you had to say no to a feature request because it wasn't safe/ready for production."* — Given the guardrails/governance emphasis in this JD, having a real example here is high-value.
- *"How do you stay current given how fast this space moves?"* — Your MTech AIML coursework + hands-on Cognizant work is a genuine, specific answer here.

---

## 10. Quick-hit gap primers (skim before the test)

- **ADK**: Google's agent framework — agent definition + tool integration + built-in eval + native Vertex AI deploy. Analogous role to LangGraph but more GCP-opinionated.
- **AlloyDB**: Postgres-compatible, GCP-managed, has a vector extension + Vertex AI embedding integration — lets you combine relational + vector queries in one database.
- **Terraform**: declarative IaC tool; you write `.tf` files describing infra, run `terraform plan`/`apply` to create it idempotently and repeatably.
- **GraphRAG**: builds an entity-relationship knowledge graph from your corpus (via LLM extraction) and retrieves by graph traversal, not just vector similarity — better for multi-hop, relationship-heavy questions.
- **LangSmith**: LangChain's observability/eval platform — full trace of every LLM call, tool call, and intermediate state in a run, plus regression-test datasets.

Good luck — given your MCP + LangGraph + multi-agent + RAG background, you're already strong on the parts of this JD that carry the most weight. The main move is closing the ADK/AlloyDB/Terraform/GraphRAG vocabulary gap enough to talk about them intelligently, even without hands-on depth.